# Task 12 & 13: GHZ state prep via cluster fusion — DynamicCircuit

Demonstrates `DynamicCircuit` and `ghz_fusion_dqc`:

1. `bell_basis_circuit` — Bell measurement pre-rotation
2. `ghz_cluster_prep_circuit` — GHZ state on offset/independent sites
3. **DQC loop step-by-step**: `ghz_fusion_dqc` structure → `dqc.run()` → inspect outcome/correction/state
4. Statistical verification: fidelity = 1.0 for all outcomes, all 4 outcomes covered
5. Multi-boundary case (n=6, cluster_size=4 clusters, 3 fusions)


In [1]:
import numpy as np
from qaravan.applications import (
    bell_basis_circuit,
    build_ghz_decoder,
    ghz_cluster_prep_circuit,
    ghz_fusion_dqc,
)
from qaravan.backends.statevector import Statevector

INV_SQRT2 = 1.0 / np.sqrt(2)


## 1. Bell measurement pre-rotation

`bell_basis_circuit(a, b, n)` = CNOT(a→b), H(a).  
Applied to the 4 Bell states it returns the 4 computational basis states.

In [2]:
bell_states = {
    "|Φ+⟩": np.array([INV_SQRT2, 0, 0, INV_SQRT2]),
    "|Φ-⟩": np.array([INV_SQRT2, 0, 0, -INV_SQRT2]),
    "|Ψ+⟩": np.array([0, INV_SQRT2, INV_SQRT2, 0]),
    "|Ψ-⟩": np.array([0, INV_SQRT2, -INV_SQRT2, 0]),
}

circ = bell_basis_circuit(0, 1, 2)
print("bell_basis_circuit(0,1,2):", circ.gates)
print()
for name, arr in bell_states.items():
    result = Statevector(array=arr).apply(circ)
    print(f"{name} → {result}")

bell_basis_circuit(0,1,2): [CNOT([0, 1]), H([0])]

|Φ+⟩ → 1.0000|00⟩
|Φ-⟩ → 1.0000|10⟩
|Ψ+⟩ → 1.0000|01⟩
|Ψ-⟩ → 1.0000|11⟩


## 2. Two independent GHZ clusters (pre-fusion)

Star-topology GHZ on offset sites. Purity of each cluster = 1 confirms they are
unentangled with each other before any fusion.

In [3]:
init6 = Statevector(bitstring="000000")
sv_clusters = init6.apply(
    ghz_cluster_prep_circuit([0, 1, 2], 6) + ghz_cluster_prep_circuit([3, 4, 5], 6)
)
print("State before fusion:")
print(sv_clusters)
print()

State before fusion:
0.5000|000000⟩ + 0.5000|000111⟩ + 0.5000|111000⟩ + 0.5000|111111⟩



## 3. DQC loop step-by-step

`ghz_fusion_dqc(n, cluster_size)` returns a `DynamicCircuit` with two rounds:
- **Round 0**: `meas_sites=[]`, constant decoder applies `prep_circuit + bell_circuit`
- **Round 1**: `meas_sites=boundary_qubits`, decoder returns Pauli correction based on outcome

We step through the rounds explicitly here to inspect each intermediate state.  
**Re-run this cell several times** — each run draws a different measurement outcome and correction.


In [5]:
n, cluster_size = 5, 3
num_clusters = (n - 2) // (cluster_size - 2)
total_qubits = num_clusters * cluster_size

dqc = ghz_fusion_dqc(n, cluster_size)
fusion_sites = dqc.rounds[1].meas_sites  # [2, 3]
kept_sites = [s for s in range(total_qubits) if s not in fusion_sites]
kept_str = "{" + ", ".join(str(s) for s in kept_sites) + "}"

print(f"DynamicCircuit: {len(dqc.rounds)} rounds")
print(f"  Round 0: meas_sites={dqc.rounds[0].meas_sites}  (prep + Bell rotation, no measurement)")
print(f"  Round 1: meas_sites={dqc.rounds[1].meas_sites}  (measure boundary qubits, apply correction)")
print()

# Step through rounds using the DQC's own components for transparency
init = Statevector(bitstring="0" * total_qubits)

# Round 0: constant decoder applies prep + Bell rotation
sv = init.apply(dqc.rounds[0].decoder(""))

# Round 1: measure boundary qubits
sv, outcome = sv.measure_and_collapse(fusion_sites)
print(f"Outcome on qubits {fusion_sites}: '{outcome}'")
print(f"Post-measurement state on kept qubits {kept_str}:")
print(sv.drop_sites(fusion_sites, outcome))
print()

# Round 1: decode outcome -> correction circuit
correction = dqc.rounds[1].decoder(outcome)
gate_str = str(correction.gates) if correction.gates else "[none needed]"
print(f"decoder('{outcome}') -> correction: {gate_str}")
print()

# Apply correction
if correction.gates:
    sv = sv.apply(correction)
print(f"Final state on kept qubits {kept_str}:")
print(sv.drop_sites(fusion_sites, outcome))
print()

ghz_target = np.zeros(2**n)
ghz_target[0] = ghz_target[-1] = INV_SQRT2
fidelity = float(abs(np.dot(ghz_target, sv.drop_sites(fusion_sites, outcome).to_array()))**2)
print(f"Fidelity with |GHZ_{n}>: {fidelity:.12f}")


DynamicCircuit: 2 rounds
  Round 0: meas_sites=[]  (prep + Bell rotation, no measurement)
  Round 1: meas_sites=[2, 3, 5, 6]  (measure boundary qubits, apply correction)

Outcome on qubits [2, 3, 5, 6]: '1001'
Post-measurement state on kept qubits {0, 1, 4, 7, 8}:
0.7071|00011⟩ + -0.7071|11100⟩

decoder('1001') -> correction: [Z([0]), X([7]), X([8])]

Final state on kept qubits {0, 1, 4, 7, 8}:
0.7071|00000⟩ + 0.7071|11111⟩

Fidelity with |GHZ_5>: 1.000000000000


## 4. Statistical verification: all outcomes, always fidelity = 1.0

In [6]:
from collections import Counter

dqc4 = ghz_fusion_dqc(4, cluster_size=3)
init4 = Statevector(bitstring="000000")
fusion_sites_4 = [2, 3]

outcome_counts = Counter()
fidelities = []
ghz4 = np.zeros(16)
ghz4[0] = ghz4[15] = INV_SQRT2

for _ in range(200):
    sv_f, outcomes = dqc4.run(init4)
    pure_kept = sv_f.drop_sites(fusion_sites_4, outcomes[0])
    fidelities.append(float(abs(np.dot(ghz4, pure_kept.to_array()))**2))
    outcome_counts[outcomes[0]] += 1

print("All fidelities = 1.0:", all(abs(f - 1.0) < 1e-9 for f in fidelities))
print("Min fidelity:          ", min(fidelities))
print()
print("Outcome distribution (200 runs):")
for o in sorted(outcome_counts):
    print(f"  {o}: {outcome_counts[o]}")


All fidelities = 1.0: True
Min fidelity:           1.0

Outcome distribution (200 runs):
  00: 51
  01: 40
  10: 58
  11: 51


## 5. Multi-boundary: 6-qubit GHZ (n=6, cluster_size=3)

4 clusters, 3 fusions, 12 physical qubits.  
Kept sites: {0,1} | {4} | {7} | {10,11}  
Fusion sites: {2,3} (boundary 0), {5,6} (boundary 1), {8,9} (boundary 2)  
Outcome string: 6 bits, pairs 0-1 / 2-3 / 4-5 for each boundary.


In [7]:
dqc6 = ghz_fusion_dqc(6, cluster_size=3)
init6 = Statevector(bitstring="0" * 12)
fusion_sites_6 = [2, 3, 5, 6, 8, 9]
ghz6 = np.zeros(64)
ghz6[0] = ghz6[63] = INV_SQRT2

fidelities6 = []
for _ in range(20):
    sv_f, outcomes = dqc6.run(init6)
    assert len(outcomes[0]) == 6  # 3 boundaries x 2 bits
    pure_kept = sv_f.drop_sites(fusion_sites_6, outcomes[0])
    fidelities6.append(float(abs(np.dot(ghz6, pure_kept.to_array()))**2))

print("All fidelities = 1.0 (n=6, 3 fusions):", all(abs(f - 1.0) < 1e-9 for f in fidelities6))
print("Example outcome string:", outcomes[0], " (pairs: boundary 0 | boundary 1 | boundary 2)")


All fidelities = 1.0 (n=6, 3 fusions): True
Example outcome string: 000011  (pairs: boundary 0 | boundary 1 | boundary 2)
